# LeetCode 1297 — Maximum Number of Occurrences of a Substring bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-occurrences-of-a-substring/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết cách **đếm tần suất substring** bằng `dict` / `collections.Counter`, kể cả khi các lần xuất hiện **chồng lấn** nhau.
2. Hiểu **insight then chốt**: chỉ cần xét substring có độ dài **đúng `minSize`** — tham số `maxSize` thực chất là "mồi nhử".
3. Thành thạo **cửa sổ trượt độ dài cố định** kèm bảng đếm ký tự để biết số ký tự phân biệt trong `O(1)` mỗi bước.
4. Làm quen **rolling hash (Rabin–Karp)**: biến mỗi cửa sổ thành một con số, cập nhật `O(1)` khi trượt.
5. Tránh các lỗi kinh điển: dùng `str.count` (đếm không chồng lấn), lệch chỉ số cửa sổ cuối, quên giảm số ký tự phân biệt, gọi `max()` trên dict rỗng.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Trả về **số lần xuất hiện lớn nhất** của một substring (chuỗi con liên tiếp) bất kỳ thoả hai điều kiện:

- Số **ký tự phân biệt** trong substring `<= maxLetters`.
- Độ dài substring nằm trong đoạn `[minSize, maxSize]`.

Các lần xuất hiện **được phép chồng lấn** (overlap). Nếu không có substring nào hợp lệ, trả về `0`.

**Ví dụ 1:** `s = "aababcaab", maxLetters = 2, minSize = 3, maxSize = 4` → `2` (substring `"aab"` xuất hiện 2 lần, có 2 ký tự phân biệt, độ dài 3).

**Ví dụ 2:** `s = "aaaa", maxLetters = 1, minSize = 3, maxSize = 3` → `2` (`"aaa"` xuất hiện ở vị trí 0 và 1 — chồng lấn vẫn tính).

**Ràng buộc:**
- `1 <= s.length <= 10^5`
- `1 <= maxLetters <= 26`
- `1 <= minSize <= maxSize <= min(26, s.length)`
- `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import time
from collections import Counter, defaultdict
from typing import List

examples = [
    (("aababcaab", 2, 3, 4), 2),
    (("aaaa", 1, 3, 3), 2),
]
for (s, ml, mn, mx), want in examples:
    print(f"s={s!r}, maxLetters={ml}, minSize={mn}, maxSize={mx} -> mong đợi {want}")

s='aababcaab', maxLetters=2, minSize=3, maxSize=4 -> mong đợi 2
s='aaaa', maxLetters=1, minSize=3, maxSize=3 -> mong đợi 2


## 3. Kiến thức nền

### 3.1 Cắt chuỗi (slicing) `s[i:j]`

`s[i:j]` lấy các ký tự từ chỉ số `i` đến `j-1`. Substring độ dài `L` bắt đầu tại `i` là `s[i:i+L]`, với `i` chạy từ `0` đến `len(s) - L` (**bao gồm**), tức `range(len(s) - L + 1)`. Chi phí cắt là `O(L)` vì Python tạo chuỗi mới.

In [2]:
s = "aababcaab"
L = 3
windows = [s[i:i + L] for i in range(len(s) - L + 1)]
print("số cửa sổ =", len(windows), "=", len(s), "-", L, "+ 1")
print(windows)

số cửa sổ = 7 = 9 - 3 + 1
['aab', 'aba', 'bab', 'abc', 'bca', 'caa', 'aab']


### 3.2 Đếm ký tự phân biệt bằng `set`

`len(set(t))` cho số ký tự khác nhau của `t`. Đơn giản nhưng tốn `O(len(t))` mỗi lần gọi.

In [3]:
for t in ["aab", "abc", "aaaa", "abab"]:
    print(t, "->", set(t), "-> số ký tự phân biệt:", len(set(t)))

aab -> {'b', 'a'} -> số ký tự phân biệt: 2
abc -> {'b', 'c', 'a'} -> số ký tự phân biệt: 3
aaaa -> {'a'} -> số ký tự phân biệt: 1
abab -> {'b', 'a'} -> số ký tự phân biệt: 2


### 3.3 `collections.Counter` và `dict.get`

`Counter` là `dict` đặc biệt để đếm: `cnt[x] += 1` không cần khởi tạo trước. `cnt.most_common(1)` trả phần tử nhiều nhất. Với `dict` thường, dùng `d[x] = d.get(x, 0) + 1`.

Lưu ý: `max(cnt.values())` sẽ **báo lỗi** nếu `cnt` rỗng — dùng `max(cnt.values(), default=0)`.

In [4]:
cnt = Counter()
for w in ["aab", "aba", "bab", "aab"]:
    cnt[w] += 1
print(cnt, "| most_common:", cnt.most_common(1))
print("max an toàn:", max(Counter().values(), default=0))

d = {}
for w in ["x", "y", "x"]:
    d[w] = d.get(w, 0) + 1
print(d)

Counter({'aab': 2, 'aba': 1, 'bab': 1}) | most_common: [('aab', 2)]
max an toàn: 0
{'x': 2, 'y': 1}


### 3.4 Bảng đếm 26 chữ cái

Vì `s` chỉ có chữ thường, ta dùng mảng 26 phần tử: chỉ số của ký tự `ch` là `ord(ch) - ord('a')`. Kèm biến `distinct` = số ô đang `> 0`. Khi một ô tăng từ 0 lên 1 thì `distinct += 1`, khi giảm từ 1 về 0 thì `distinct -= 1`.

In [5]:
freq = [0] * 26
distinct = 0
for ch in "abca":
    k = ord(ch) - ord('a')
    if freq[k] == 0:
        distinct += 1
    freq[k] += 1
print("freq a,b,c =", freq[:3], "| distinct =", distinct)

freq a,b,c = [2, 1, 1] | distinct = 3


### 3.5 Băm đa thức (polynomial hash)

Coi chuỗi như một số ở hệ cơ số `B`: `h("abc") = a·B² + b·B + c` (mod `M`). Khi trượt cửa sổ sang phải: bỏ ký tự đầu (trừ `first·B^(L-1)`), nhân `B`, cộng ký tự mới. Python có `pow(B, L-1, M)` tính lũy thừa theo modulo nhanh.

In [6]:
B, M = 131, (1 << 61) - 1
def poly_hash(t):
    h = 0
    for ch in t:
        h = (h * B + ord(ch)) % M
    return h

s, L = "abcd", 3
h = poly_hash(s[:L])
top = pow(B, L - 1, M)
h2 = ((h - ord(s[0]) * top) * B + ord(s[L])) % M
print("hash('bcd') trực tiếp:", poly_hash("bcd"))
print("hash('bcd') trượt     :", h2)

hash('bcd') trực tiếp: 1694847
hash('bcd') trượt     : 1694847


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `s = "aababcaab"`, `maxLetters = 2`, `minSize = 3`, `maxSize = 4`.

Liệt kê các substring độ dài 3 (7 cửa sổ):

| i | s[i:i+3] | phân biệt | hợp lệ? |
|---|---|---|---|
| 0 | aab | 2 | ✔ |
| 1 | aba | 2 | ✔ |
| 2 | bab | 2 | ✔ |
| 3 | abc | 3 | ✘ |
| 4 | bca | 3 | ✘ |
| 5 | caa | 2 | ✔ |
| 6 | aab | 2 | ✔ |

Đếm các cửa sổ hợp lệ: `aab` ×2, `aba` ×1, `bab` ×1, `caa` ×1 ⇒ max = **2**.

Còn độ dài 4? Các substring: `aaba, abab, babc, abca, bcaa, caab` — mỗi cái chỉ xuất hiện 1 lần (và vài cái có 3 ký tự phân biệt). Không cải thiện được. Nhận xét đáng chú ý: **độ dài lớn hơn không bao giờ thắng** — ta sẽ chứng minh ở mục 6.

In [7]:
s, ml = "aababcaab", 2
for L in (3, 4):
    cnt = Counter(s[i:i + L] for i in range(len(s) - L + 1) if len(set(s[i:i + L])) <= ml)
    print(f"L={L}:", dict(cnt), "-> max", max(cnt.values(), default=0))

L=3: {'aab': 2, 'aba': 1, 'bab': 1, 'caa': 1} -> max 2
L=4: {'aaba': 1, 'abab': 1} -> max 1


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** làm đúng như đề nói. Với mỗi độ dài `L` từ `minSize` đến `maxSize`, với mỗi vị trí bắt đầu `i`, lấy `t = s[i:i+L]`; nếu `len(set(t)) <= maxLetters` thì tăng `cnt[t]`. Cuối cùng trả `max` của các giá trị đếm (hoặc 0).

Vì `maxSize <= 26`, số độ dài cần thử tối đa 26 — cách này **thực ra vẫn qua được** LeetCode (đúng như gợi ý của đề), nhưng chậm hơn cần thiết.

In [8]:
def max_freq_brute(s, maxLetters, minSize, maxSize):
    cnt = Counter()
    n = len(s)
    for L in range(minSize, maxSize + 1):
        for i in range(n - L + 1):
            t = s[i:i + L]
            if len(set(t)) <= maxLetters:
                cnt[t] += 1
    return max(cnt.values(), default=0)

for args, want in examples:
    print(args, "->", max_freq_brute(*args), "| mong đợi", want)

('aababcaab', 2, 3, 4) -> 2 | mong đợi 2
('aaaa', 1, 3, 3) -> 2 | mong đợi 2


**Độ phức tạp:** gọi `n = len(s)`, `K = maxSize - minSize + 1` (≤ 26), `m = maxSize`.
- Time: `O(n · K · m)` — mỗi cửa sổ tốn `O(m)` để cắt và tạo `set`. Với `n = 10^5`, `K = m = 26` ⇒ khoảng `6.8·10^7` thao tác ký tự: chạy được nhưng chậm.
- Space: `O(n · K · m)` trong trường hợp xấu — `Counter` có thể lưu tới `n·K` chuỗi khác nhau, mỗi chuỗi dài tới `m`.

**Nút thắt:** ta đếm cả những substring dài mà **không bao giờ** cho kết quả tốt hơn substring ngắn. Đó là chỗ lãng phí cần cắt bỏ.

In [9]:
rng = random.Random(1)
big = "".join(rng.choice("ab") for _ in range(20000))
t0 = time.perf_counter()
r1 = max_freq_brute(big, 2, 3, 26)
print("brute trên n=20000, K=24:", r1, f"({time.perf_counter() - t0:.2f}s)")

brute trên n=20000, K=24: 2569 (0.24s)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** *Nếu một substring `t` dài `L > minSize` hợp lệ và xuất hiện `k` lần, thì **tiền tố** độ dài `minSize` của nó cũng hợp lệ và xuất hiện **ít nhất** `k` lần.*

Lý do:
1. **Số lần xuất hiện:** mỗi lần `t` xuất hiện tại vị trí `i`, tiền tố `t[:minSize]` cũng xuất hiện tại đúng vị trí `i`. Các vị trí khác nhau ⇒ tiền tố có ít nhất `k` lần (có thể nhiều hơn).
2. **Số ký tự phân biệt:** tiền tố là một phần của `t` nên số ký tự phân biệt **không tăng** ⇒ vẫn `<= maxLetters`.
3. **Độ dài:** `minSize` nằm trong `[minSize, maxSize]` ⇒ hợp lệ.

**Quy tắc "nếu… thì…":**
- *Nếu* đáp án đạt được bởi một substring dài hơn `minSize`, *thì* cũng đạt được bởi substring độ dài `minSize`.
- ⇒ **Bỏ qua hoàn toàn `maxSize`**, chỉ xét cửa sổ độ dài cố định `minSize`.

**Invariant cho cửa sổ trượt:** khi cửa sổ là `s[i : i+minSize]`, mảng `freq` chứa đúng tần suất từng chữ trong cửa sổ, và `distinct` = số chữ có `freq > 0`.

In [10]:
# Minh hoạ insight: mỗi lần xuất hiện của t kéo theo một lần xuất hiện của tiền tố
s = "abababab"
t, minSize = "ababa", 3
pos_t = [i for i in range(len(s) - len(t) + 1) if s.startswith(t, i)]
pre = t[:minSize]
pos_p = [i for i in range(len(s) - minSize + 1) if s.startswith(pre, i)]
print(f"{t!r} xuất hiện tại {pos_t} ({len(pos_t)} lần)")
print(f"tiền tố {pre!r} xuất hiện tại {pos_p} ({len(pos_p)} lần) ⊇ các vị trí trên")

'ababa' xuất hiện tại [0, 2] (2 lần)
tiền tố 'aba' xuất hiện tại [0, 2, 4] (3 lần) ⊇ các vị trí trên


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chỉ xét độ dài `minSize` + `Counter`

Áp dụng ngay insight: chỉ một vòng lặp qua các vị trí `i`, cắt `s[i:i+minSize]`, kiểm tra bằng `set`. Ngắn gọn và đủ nhanh.

Từng đoạn:
- `range(n - minSize + 1)`: tất cả cửa sổ độ dài `minSize`.
- `len(set(t)) <= maxLetters`: điều kiện ký tự phân biệt.
- `max(..., default=0)`: an toàn khi không cửa sổ nào hợp lệ.

In [11]:
def max_freq_min_only(s, maxLetters, minSize, maxSize):
    cnt = Counter()
    for i in range(len(s) - minSize + 1):
        t = s[i:i + minSize]
        if len(set(t)) <= maxLetters:
            cnt[t] += 1
    return max(cnt.values(), default=0)

for args, want in examples:
    print(args, "->", max_freq_min_only(*args), "| mong đợi", want)

('aababcaab', 2, 3, 4) -> 2 | mong đợi 2
('aaaa', 1, 3, 3) -> 2 | mong đợi 2


Code nộp LeetCode (cách 2):

In [12]:
class SolutionMinOnly:
    def maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int:
        cnt = Counter()
        for i in range(len(s) - minSize + 1):
            t = s[i:i + minSize]
            if len(set(t)) <= maxLetters:
                cnt[t] += 1
        return max(cnt.values(), default=0)

print(SolutionMinOnly().maxFreq("aababcaab", 2, 3, 4))

2


**Độ phức tạp cách 2:** Time `O(n · minSize)` (mỗi cửa sổ cắt + tạo set `O(minSize)`), Space `O(n · minSize)` cho `Counter`. So với brute force đã bớt hẳn một thừa số `K`.

### 7.2 Cách 3 — Cửa sổ trượt với bảng đếm ký tự

Việc tạo `set` cho mỗi cửa sổ là lặp lại công sức: hai cửa sổ liền nhau chỉ khác **một ký tự vào** và **một ký tự ra**. Ta duy trì `freq[26]` và `distinct`:

1. Thêm `s[r]` vào cửa sổ: nếu `freq` từ 0 → 1 thì `distinct += 1`.
2. Nếu cửa sổ dài hơn `minSize`: bỏ `s[r - minSize]`; nếu `freq` từ 1 → 0 thì `distinct -= 1`.
3. Khi cửa sổ đủ dài `minSize` và `distinct <= maxLetters`: đếm `s[r-minSize+1 : r+1]`.

Kiểm tra điều kiện giờ là `O(1)`; chỉ còn chi phí cắt chuỗi để làm khoá cho `Counter` (chỉ cho cửa sổ hợp lệ).

In [13]:
def max_freq_sliding(s, maxLetters, minSize, maxSize):
    freq = [0] * 26
    distinct = 0
    cnt = Counter()
    for r, ch in enumerate(s):
        k = ord(ch) - 97
        if freq[k] == 0:
            distinct += 1
        freq[k] += 1
        if r >= minSize:                       # cửa sổ vượt quá minSize -> bỏ ký tự trái
            k_out = ord(s[r - minSize]) - 97
            freq[k_out] -= 1
            if freq[k_out] == 0:
                distinct -= 1
        if r >= minSize - 1 and distinct <= maxLetters:
            cnt[s[r - minSize + 1:r + 1]] += 1
    return max(cnt.values(), default=0)

for args, want in examples:
    print(args, "->", max_freq_sliding(*args), "| mong đợi", want)

('aababcaab', 2, 3, 4) -> 2 | mong đợi 2
('aaaa', 1, 3, 3) -> 2 | mong đợi 2


Code nộp LeetCode (cách 3):

In [14]:
class SolutionSliding:
    def maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int:
        freq = [0] * 26
        distinct = 0
        cnt = {}
        best = 0
        for r, ch in enumerate(s):
            k = ord(ch) - 97
            if freq[k] == 0:
                distinct += 1
            freq[k] += 1
            if r >= minSize:
                k_out = ord(s[r - minSize]) - 97
                freq[k_out] -= 1
                if freq[k_out] == 0:
                    distinct -= 1
            if r >= minSize - 1 and distinct <= maxLetters:
                t = s[r - minSize + 1:r + 1]
                cnt[t] = cnt.get(t, 0) + 1
                best = max(best, cnt[t])
        return best

print(SolutionSliding().maxFreq("aaaa", 1, 3, 3))

2


**Độ phức tạp cách 3:** Time `O(n · minSize)` trường hợp xấu (do cắt chuỗi làm khoá), nhưng hằng số nhỏ hơn cách 2 vì không tạo `set`; phần kiểm tra điều kiện chỉ `O(n)`. Space `O(n · minSize)`.

### 7.3 Cách 4 — Cửa sổ trượt + rolling hash (Rabin–Karp)

Muốn loại bỏ cả chi phí cắt chuỗi, ta thay khoá chuỗi bằng **giá trị băm** cập nhật `O(1)`:

```
h_mới = ((h_cũ − s[r−L]·B^(L−1)) · B + s[r]) mod M
```

Dùng `M = 2^61 − 1` (số nguyên tố Mersenne) và cơ số `B` lớn để xác suất va chạm cực nhỏ (`~ n² / M ≈ 10^-8`). Với bài chỉ hỏi **số lần**, ta đếm theo hash. Lưu ý: đây là thuật toán **xác suất** — trên thực tế an toàn, và ta sẽ đối chiếu với brute force trên hàng nghìn input ngẫu nhiên.

In [15]:
def max_freq_rolling(s, maxLetters, minSize, maxSize):
    B, M = 911382323, (1 << 61) - 1
    L = minSize
    top = pow(B, L - 1, M)                    # B^(L-1) để bỏ ký tự đầu
    freq = [0] * 26
    distinct = 0
    h = 0
    cnt = defaultdict(int)
    best = 0
    for r, ch in enumerate(s):
        c = ord(ch) - 96                      # 1..26 (tránh giá trị 0)
        if r >= L:                            # bỏ ký tự trái trước khi thêm ký tự mới
            out = s[r - L]
            co = ord(out) - 96
            h = (h - co * top) % M
            freq[co - 1] -= 1
            if freq[co - 1] == 0:
                distinct -= 1
        h = (h * B + c) % M
        if freq[c - 1] == 0:
            distinct += 1
        freq[c - 1] += 1
        if r >= L - 1 and distinct <= maxLetters:
            cnt[h] += 1
            if cnt[h] > best:
                best = cnt[h]
    return best

for args, want in examples:
    print(args, "->", max_freq_rolling(*args), "| mong đợi", want)

('aababcaab', 2, 3, 4) -> 2 | mong đợi 2
('aaaa', 1, 3, 3) -> 2 | mong đợi 2


Code nộp LeetCode (cách 4):

In [16]:
class SolutionRolling:
    def maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int:
        B, M = 911382323, (1 << 61) - 1
        L = minSize
        top = pow(B, L - 1, M)
        freq = [0] * 27
        distinct = 0
        h = 0
        cnt = {}
        best = 0
        for r, ch in enumerate(s):
            c = ord(ch) - 96
            if r >= L:
                co = ord(s[r - L]) - 96
                h = (h - co * top) % M
                freq[co] -= 1
                if freq[co] == 0:
                    distinct -= 1
            h = (h * B + c) % M
            if freq[c] == 0:
                distinct += 1
            freq[c] += 1
            if r >= L - 1 and distinct <= maxLetters:
                cnt[h] = cnt.get(h, 0) + 1
                if cnt[h] > best:
                    best = cnt[h]
        return best

print(SolutionRolling().maxFreq("aababcaab", 2, 3, 4))

2


**Độ phức tạp cách 4:** Time `O(n)` (mỗi ký tự vào/ra một lần, mọi thao tác `O(1)`), Space `O(n)` cho bảng đếm hash (tối đa `n` số nguyên). Đây là cách tối ưu về lý thuyết; trong Python, cách 2/3 thường nhanh ngang vì cắt chuỗi được viết bằng C và `minSize <= 26`.

In [17]:
rng = random.Random(7)
big = "".join(rng.choice("abc") for _ in range(100000))
for name, f in [("min_only", max_freq_min_only), ("sliding", max_freq_sliding), ("rolling", max_freq_rolling)]:
    t0 = time.perf_counter()
    r = f(big, 2, 10, 26)
    print(f"{name:9s} -> {r:5d}  ({time.perf_counter() - t0:.3f}s)")

min_only  ->     8  (0.023s)
sliding   ->     8  (0.018s)
rolling   ->     8  (0.047s)


## 8. TRACE từng bước

Trace cách 3 (cửa sổ trượt) trên Ví dụ 1: `s = "aababcaab"`, `maxLetters = 2`, `minSize = 3`.

| r | s[r] vào | s[r−3] ra | cửa sổ | distinct | hợp lệ? | cnt sau bước |
|---|---|---|---|---|---|---|
| 0 | a | – | a | 1 | (chưa đủ dài) | {} |
| 1 | a | – | aa | 1 | (chưa đủ dài) | {} |
| 2 | b | – | aab | 2 | ✔ | aab:1 |
| 3 | a | a | aba | 2 | ✔ | aba:1 |
| 4 | b | a | bab | 2 | ✔ | bab:1 |
| 5 | c | b | abc | 3 | ✘ | – |
| 6 | a | a | bca | 3 | ✘ | – |
| 7 | a | b | caa | 2 | ✔ | caa:1 |
| 8 | b | c | aab | 2 | ✔ | aab:**2** |

Kết quả: `max = 2`.

In [18]:
def trace_sliding(s, maxLetters, minSize):
    freq = [0] * 26
    distinct = 0
    cnt = Counter()
    for r, ch in enumerate(s):
        k = ord(ch) - 97
        if freq[k] == 0:
            distinct += 1
        freq[k] += 1
        out = "-"
        if r >= minSize:
            out = s[r - minSize]
            ko = ord(out) - 97
            freq[ko] -= 1
            if freq[ko] == 0:
                distinct -= 1
        lo = max(0, r - minSize + 1)
        win = s[lo:r + 1]
        status = "chưa đủ dài"
        if r >= minSize - 1:
            if distinct <= maxLetters:
                cnt[win] += 1
                status = f"hợp lệ -> cnt[{win}]={cnt[win]}"
            else:
                status = "vượt maxLetters"
        print(f"r={r} vào={ch} ra={out} cửa sổ={win:<4} distinct={distinct} | {status}")
    ans = max(cnt.values(), default=0)
    print("=> đáp án:", ans)
    return ans

trace_sliding("aababcaab", 2, 3)

r=0 vào=a ra=- cửa sổ=a    distinct=1 | chưa đủ dài
r=1 vào=a ra=- cửa sổ=aa   distinct=1 | chưa đủ dài
r=2 vào=b ra=- cửa sổ=aab  distinct=2 | hợp lệ -> cnt[aab]=1
r=3 vào=a ra=a cửa sổ=aba  distinct=2 | hợp lệ -> cnt[aba]=1
r=4 vào=b ra=a cửa sổ=bab  distinct=2 | hợp lệ -> cnt[bab]=1
r=5 vào=c ra=b cửa sổ=abc  distinct=3 | vượt maxLetters
r=6 vào=a ra=a cửa sổ=bca  distinct=3 | vượt maxLetters
r=7 vào=a ra=b cửa sổ=caa  distinct=2 | hợp lệ -> cnt[caa]=1
r=8 vào=b ra=c cửa sổ=aab  distinct=2 | hợp lệ -> cnt[aab]=2
=> đáp án: 2


2

In [19]:
trace_sliding("aaaa", 1, 3)

r=0 vào=a ra=- cửa sổ=a    distinct=1 | chưa đủ dài
r=1 vào=a ra=- cửa sổ=aa   distinct=1 | chưa đủ dài
r=2 vào=a ra=- cửa sổ=aaa  distinct=1 | hợp lệ -> cnt[aaa]=1
r=3 vào=a ra=a cửa sổ=aaa  distinct=1 | hợp lệ -> cnt[aaa]=2
=> đáp án: 2


2

## 9. Code nộp LeetCode

Bản chính thức: cửa sổ trượt độ dài `minSize` + bảng đếm ký tự + `Counter` theo chuỗi (dễ đọc, không lo va chạm hash, đủ nhanh vì `minSize <= 26`).

In [20]:
class Solution:
    def maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int:
        # Chỉ cần xét độ dài minSize (tiền tố của chuỗi dài hơn luôn tốt bằng hoặc hơn)
        freq = [0] * 26
        distinct = 0
        cnt = {}
        best = 0
        for r, ch in enumerate(s):
            k = ord(ch) - 97
            if freq[k] == 0:
                distinct += 1
            freq[k] += 1
            if r >= minSize:
                ko = ord(s[r - minSize]) - 97
                freq[ko] -= 1
                if freq[ko] == 0:
                    distinct -= 1
            if r >= minSize - 1 and distinct <= maxLetters:
                t = s[r - minSize + 1:r + 1]
                cnt[t] = cnt.get(t, 0) + 1
                if cnt[t] > best:
                    best = cnt[t]
        return best

sol = Solution()
for args, want in examples:
    got = sol.maxFreq(*args)
    print(args, "->", got, "✔" if got == want else "✘")

('aababcaab', 2, 3, 4) -> 2 ✔
('aaaa', 1, 3, 3) -> 2 ✔


## 10. Vì sao thuật toán đúng

Gọi `OPT` là đáp án thật (max trên mọi độ dài trong `[minSize, maxSize]`) và `A` là max chỉ trên độ dài `minSize`.

1. **`A <= OPT`:** các substring độ dài `minSize` là một phần của tập được xét trong `OPT`.
2. **`OPT <= A`:** lấy substring `t` đạt `OPT` với `k` lần xuất hiện tại các vị trí `i_1 < … < i_k`. Tiền tố `p = t[:minSize]` xuất hiện tại cùng các vị trí ⇒ ít nhất `k` lần; `p` có số ký tự phân biệt `<=` của `t` `<= maxLetters` ⇒ `p` hợp lệ ⇒ `A >= k = OPT`.
3. Vậy `A = OPT`.

**Đúng đắn của cửa sổ trượt:** bằng quy nạp theo `r`, sau khi xử lý bước `r`, `freq` là tần suất trong `s[max(0, r−minSize+1) : r+1]` (thêm `s[r]`, bỏ `s[r−minSize]` khi có) và `distinct` cập nhật đúng khi một ô chuyển 0↔1. Mỗi cửa sổ đủ dài được kiểm tra đúng một lần, nên `cnt` đếm chính xác số vị trí của từng substring hợp lệ — kể cả các lần chồng lấn.

## 11. Phân tích Time / Space

Gọi `n = len(s)`, `m = minSize`, `K = maxSize − minSize + 1`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force mọi độ dài | `O(n·K·m)` | `O(n·K·m)` | qua được nhờ `K, m <= 26` |
| 2. Chỉ `minSize` + set | `O(n·m)` | `O(n·m)` | đơn giản nhất |
| 3. Cửa sổ trượt + Counter | `O(n·m)` worst (cắt chuỗi), kiểm tra `O(n)` | `O(n·m)` | bản nộp chính |
| 4. Rolling hash | `O(n)` | `O(n)` | xác suất va chạm ~`10^-8` |

Trường hợp xấu cho Space là khi mọi cửa sổ đều khác nhau và đều hợp lệ (ví dụ `maxLetters = 26`, chuỗi ngẫu nhiên). Bảng `freq` chỉ tốn `O(26) = O(1)`.

## 12. Bộ test

Gồm: ví dụ đề, edge case (chuỗi 1 ký tự, `minSize = len(s)`, không cửa sổ nào hợp lệ, `maxLetters = 26`), và đối chiếu 4 cách trên input ngẫu nhiên có seed cố định.

In [21]:
ALL = [max_freq_brute, max_freq_min_only, max_freq_sliding, max_freq_rolling,
       Solution().maxFreq, SolutionMinOnly().maxFreq, SolutionSliding().maxFreq, SolutionRolling().maxFreq]

tests = [
    (("aababcaab", 2, 3, 4), 2),
    (("aaaa", 1, 3, 3), 2),
    (("a", 1, 1, 1), 1),                 # chuỗi 1 ký tự
    (("abc", 3, 3, 3), 1),               # minSize = len(s): đúng 1 cửa sổ
    (("abcde", 2, 3, 3), 0),             # không cửa sổ nào hợp lệ
    (("abcabc", 3, 3, 3), 2),            # "abc" xuất hiện 2 lần
    (("aabcabcab", 2, 2, 3), 3),         # "ab" xuất hiện 3 lần
    (("abcdefghijklmnopqrstuvwxyz", 26, 1, 26), 1),
    (("bbbbbb", 1, 1, 6), 6),            # độ dài 1 thắng
]
for args, want in tests:
    for f in ALL:
        got = f(*args)
        assert got == want, (f, args, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 9 test cố định qua cho 8 hàm


In [22]:
rng = random.Random(2024)
cnt_cases = 0
for _ in range(1500):
    n = rng.randint(1, 30)
    alpha = "abcdefghijklmnopqrstuvwxyz"[:rng.randint(1, 5)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    mn = rng.randint(1, min(26, n))
    mx = rng.randint(mn, min(26, n))
    ml = rng.randint(1, 26)
    want = max_freq_brute(s, ml, mn, mx)
    for f in ALL[1:]:
        assert f(s, ml, mn, mx) == want, (f, s, ml, mn, mx)
    cnt_cases += 1
print("Đối chiếu ngẫu nhiên:", cnt_cases, "bộ test đều khớp brute force")

Đối chiếu ngẫu nhiên: 1500 bộ test đều khớp brute force


In [23]:
# Kiểm tra tốc độ ở kích thước tối đa n = 10^5
rng = random.Random(99)
s_big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
t0 = time.perf_counter()
r = Solution().maxFreq(s_big, 26, 26, 26)
print("n=1e5, mọi cửa sổ khác nhau ->", r, f"({time.perf_counter() - t0:.3f}s)")
t0 = time.perf_counter()
r = Solution().maxFreq("a" * 10**5, 1, 1, 26)
print("n=1e5, toàn 'a' ->", r, f"({time.perf_counter() - t0:.3f}s)")

n=1e5, mọi cửa sổ khác nhau -> 1 (0.088s)
n=1e5, toàn 'a' -> 100000 (0.055s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `str.count` để đếm số lần xuất hiện

`s.count(t)` đếm các lần xuất hiện **không chồng lấn**. Đề lại cho phép chồng lấn.

In [24]:
def wrong_count(s, maxLetters, minSize, maxSize):
    best = 0
    for i in range(len(s) - minSize + 1):
        t = s[i:i + minSize]
        if len(set(t)) <= maxLetters:
            best = max(best, s.count(t))     # SAI: không đếm chồng lấn
    return best

print("sai :", wrong_count("aaaa", 1, 3, 3), "| đúng:", Solution().maxFreq("aaaa", 1, 3, 3))

sai : 1 | đúng: 2


**Vì sao sai:** `"aaaa".count("aaa") == 1` vì sau khi khớp ở vị trí 0, Python nhảy tới vị trí 3. **Sửa:** đếm theo từng vị trí bắt đầu bằng `Counter` như cách 2/3 (ngoài ra `s.count` trong vòng lặp còn làm thuật toán thành `O(n²)`).

In [25]:
def fixed_count(s, maxLetters, minSize, maxSize):
    cnt = Counter(s[i:i + minSize] for i in range(len(s) - minSize + 1)
                  if len(set(s[i:i + minSize])) <= maxLetters)
    return max(cnt.values(), default=0)

print("đã sửa:", fixed_count("aaaa", 1, 3, 3))

đã sửa: 2


### Lỗi 2 — Lệch chỉ số: bỏ sót cửa sổ cuối

Viết `range(len(s) - minSize)` thay vì `range(len(s) - minSize + 1)`.

In [26]:
def wrong_range(s, maxLetters, minSize, maxSize):
    cnt = Counter()
    for i in range(len(s) - minSize):        # SAI: thiếu + 1
        t = s[i:i + minSize]
        if len(set(t)) <= maxLetters:
            cnt[t] += 1
    return max(cnt.values(), default=0)

print("s='abc', minSize=3 -> sai:", wrong_range("abc", 3, 3, 3), "| đúng:", Solution().maxFreq("abc", 3, 3, 3))
print("Ví dụ 1           -> sai:", wrong_range("aababcaab", 2, 3, 4), "| đúng:", Solution().maxFreq("aababcaab", 2, 3, 4))

s='abc', minSize=3 -> sai: 0 | đúng: 1
Ví dụ 1           -> sai: 1 | đúng: 2


**Vì sao sai:** có `n − L + 1` cửa sổ độ dài `L`; thiếu `+1` làm mất cửa sổ cuối (trong Ví dụ 1 đó chính là lần xuất hiện thứ hai của `"aab"`). **Sửa:** `range(len(s) - minSize + 1)` — mẹo nhớ: khi `L = n` phải có đúng 1 cửa sổ.

### Lỗi 3 — Cửa sổ trượt quên giảm `distinct`

Bỏ ký tự trái nhưng không kiểm tra `freq` về 0 ⇒ `distinct` chỉ tăng mãi.

In [27]:
def wrong_distinct(s, maxLetters, minSize, maxSize):
    freq = [0] * 26
    distinct = 0
    cnt = Counter()
    for r, ch in enumerate(s):
        k = ord(ch) - 97
        if freq[k] == 0:
            distinct += 1
        freq[k] += 1
        if r >= minSize:
            freq[ord(s[r - minSize]) - 97] -= 1   # SAI: quên distinct -= 1 khi về 0
        if r >= minSize - 1 and distinct <= maxLetters:
            cnt[s[r - minSize + 1:r + 1]] += 1
    return max(cnt.values(), default=0)

print("sai :", wrong_distinct("aababcaab", 2, 3, 4), "| đúng:", Solution().maxFreq("aababcaab", 2, 3, 4))

sai : 1 | đúng: 2


**Vì sao sai:** sau khi gặp `'c'`, `distinct` = 3 mãi mãi nên cửa sổ `"caa"`, `"aab"` cuối bị loại oan ⇒ kết quả 1 thay vì 2. **Sửa:** sau `freq[ko] -= 1` thêm `if freq[ko] == 0: distinct -= 1` (xem `Solution` ở mục 9).

### Lỗi 4 — `max()` trên `Counter` rỗng

Khi không có cửa sổ hợp lệ, `max(cnt.values())` ném `ValueError`.

In [28]:
def wrong_empty(s, maxLetters, minSize, maxSize):
    cnt = Counter(s[i:i + minSize] for i in range(len(s) - minSize + 1)
                  if len(set(s[i:i + minSize])) <= maxLetters)
    return max(cnt.values())                 # SAI khi cnt rỗng

try:
    print(wrong_empty("abcde", 2, 3, 3))
except ValueError as e:
    print("Lỗi bắt được:", type(e).__name__, "-", e)
print("đúng:", Solution().maxFreq("abcde", 2, 3, 4))

Lỗi bắt được: ValueError - max() iterable argument is empty
đúng: 0


**Sửa:** `max(cnt.values(), default=0)` hoặc duy trì biến `best = 0` và cập nhật dần.

### Lỗi 5 — Rolling hash không lấy modulo sau phép trừ (ngôn ngữ khác) / dùng mã 0 cho `'a'`

Trong Python `%` luôn trả số không âm nên phép trừ an toàn. Nhưng nếu mã hoá `'a'` thành 0, các chuỗi `"a"`, `"aa"`, `"aaa"` đều có hash 0 — với cửa sổ cố định thì không sao, nhưng là thói quen nguy hiểm khi so chuỗi khác độ dài. Luôn dùng mã từ 1 (`ord(ch) - 96`).

In [29]:
def h0(t, B=131):
    h = 0
    for ch in t:
        h = h * B + (ord(ch) - 97)     # 'a' -> 0
    return h
print("mã từ 0:", h0("a"), h0("aa"), h0("aaa"), "-> trùng nhau!")
def h1(t, B=131):
    h = 0
    for ch in t:
        h = h * B + (ord(ch) - 96)     # 'a' -> 1
    return h
print("mã từ 1:", h1("a"), h1("aa"), h1("aaa"), "-> khác nhau")

mã từ 0: 0 0 0 -> trùng nhau!
mã từ 1: 1 132 17293 -> khác nhau


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Thấy cặp `minSize`/`maxSize` + đếm số lần xuất hiện** ⇒ hỏi ngay: "chuỗi dài hơn có lợi gì không?". Với các điều kiện **đơn điệu theo chuỗi con** (tiền tố của chuỗi hợp lệ cũng hợp lệ) thì **ngắn nhất là tốt nhất**.
- **Cửa sổ độ dài cố định** ⇒ cập nhật bằng "1 vào, 1 ra"; giữ `freq` + `distinct`.
- **"Số ký tự phân biệt ≤ k"** ⇒ mảng 26 + biến đếm ô khác 0.
- **Cần so/đếm nhiều chuỗi con cùng độ dài** ⇒ rolling hash nếu muốn `O(n)`.
- **"Có thể chồng lấn"** ⇒ cảnh giác với `str.count`, `re.findall` (không chồng lấn).
- Đọc **gợi ý trong ràng buộc**: `maxSize <= 26` ngụ ý brute force theo độ dài vẫn qua.

## 15. Biến thể của bài

1. **Trả về chính substring** (thay vì số lần), nếu hoà thì chọn nhỏ nhất theo từ điển.
2. **Điều kiện "ít nhất `k` ký tự phân biệt"**: insight tiền tố **không còn đúng** (tiền tố có thể có ít ký tự hơn) ⇒ phải xét mọi độ dài.
3. **Đếm không chồng lấn**: với mỗi substring, đếm tham lam từ trái sang (bỏ qua các vị trí chồng lấn).
4. **LeetCode 187 — Repeated DNA Sequences**: cửa sổ độ dài 10 cố định, tìm chuỗi xuất hiện > 1 lần (rolling hash/bitmask).
5. **LeetCode 1044 — Longest Duplicate Substring**: binary search độ dài + rolling hash.
6. **LeetCode 340/159 — Longest substring with at most k distinct characters**: cửa sổ trượt độ dài **thay đổi**, cùng kỹ thuật `freq + distinct`.

In [30]:
# Biến thể 2 minh hoạ: điều kiện "ít nhất k ký tự phân biệt" -> chỉ xét minSize là SAI
def at_least_brute(s, k, mn, mx):
    cnt = Counter()
    for L in range(mn, mx + 1):
        for i in range(len(s) - L + 1):
            t = s[i:i + L]
            if len(set(t)) >= k:
                cnt[t] += 1
    return max(cnt.values(), default=0)

def at_least_min_only(s, k, mn, mx):
    cnt = Counter(s[i:i + mn] for i in range(len(s) - mn + 1) if len(set(s[i:i + mn])) >= k)
    return max(cnt.values(), default=0)

s = "aabaab"
print("s =", s, "| k = 2, độ dài 1..3")
print("  xét mọi độ dài :", at_least_brute(s, 2, 1, 3), "(ví dụ 'ab' xuất hiện 2 lần)")
print("  chỉ xét minSize:", at_least_min_only(s, 2, 1, 3), "(chuỗi độ dài 1 chỉ có 1 ký tự -> không đạt) => SAI")

s = aabaab | k = 2, độ dài 1..3
  xét mọi độ dài : 2 (ví dụ 'ab' xuất hiện 2 lần)
  chỉ xét minSize: 0 (chuỗi độ dài 1 chỉ có 1 ký tự -> không đạt) => SAI


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mọi độ dài `minSize..maxSize`, `set` + `Counter` | `O(n·K·m)` | `O(n·K·m)` | Kiểm chứng, phỏng vấn bước đầu |
| 2. Chỉ `minSize` | Insight tiền tố, `set` + `Counter` | `O(n·m)` | `O(n·m)` | Viết nhanh, ít bug |
| 3. Cửa sổ trượt | `freq[26]` + `distinct`, Counter theo chuỗi | `O(n·m)` (cắt chuỗi) | `O(n·m)` | Bản nộp chuẩn, dễ giải thích |
| 4. Rolling hash | Băm đa thức trượt `O(1)` | `O(n)` | `O(n)` | Khi `m` lớn hoặc cần tối ưu lý thuyết |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().maxFreq("abcabc", 3, 3, 3)` trả về bao nhiêu? Còn `maxFreq("abcabc", 2, 3, 3)`?

**Bài 2 (tính tay):** Với `s = "aabaab"`, `maxLetters = 1`, `minSize = 2`, `maxSize = 3`, liệt kê các cửa sổ độ dài 2 hợp lệ và cho đáp án.

**Bài 3 (điền TODO):** Hoàn thành hàm `count_distinct_windows(s, L)` trả về list số ký tự phân biệt của từng cửa sổ độ dài `L` bằng cửa sổ trượt (không dùng `set` cho từng cửa sổ).

**Bài 4 (tự viết lại):** Viết `best_substring(s, maxLetters, minSize, maxSize)` trả về **chính substring** đạt số lần lớn nhất (hoà thì nhỏ nhất theo từ điển; không có thì `""`).

**Bài 5 (mini challenge):** Viết `max_freq_at_least(s, k, minSize, maxSize)`: như đề gốc nhưng điều kiện là **ít nhất** `k` ký tự phân biệt. Gợi ý: insight cũ không dùng được.

In [31]:
# Bài 3 — khung để bạn tự điền (KHÔNG gọi khi chạy notebook)
def count_distinct_windows_todo(s, L):
    freq = [0] * 26
    distinct = 0
    res = []
    for r, ch in enumerate(s):
        # TODO: thêm s[r] vào cửa sổ, cập nhật distinct
        # TODO: nếu r >= L, bỏ s[r - L], cập nhật distinct
        # TODO: nếu r >= L - 1, res.append(distinct)
        pass
    return res

In [32]:
# Bài 4, 5 — khung (KHÔNG gọi khi chạy notebook)
def best_substring_todo(s, maxLetters, minSize, maxSize):
    # TODO
    pass

def max_freq_at_least_todo(s, k, minSize, maxSize):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Liệt kê 4 cửa sổ độ dài 3. (2) Với `maxLetters = 3` mọi cửa sổ hợp lệ. (3) Với `maxLetters = 2` mọi cửa sổ đều có 3 chữ khác nhau.

**Bài 2:** (1) Có 5 cửa sổ độ dài 2. (2) Chỉ `"aa"` có 1 ký tự phân biệt. (3) `"aa"` ở vị trí 0 và 3.

**Bài 3:** (1) Giống hệt phần cập nhật `freq/distinct` trong `Solution`. (2) Thêm trước, bỏ sau. (3) Chỉ `append` khi cửa sổ đã đủ `L` ký tự.

**Bài 4:** (1) Insight vẫn đúng: tiền tố độ dài `minSize` tốt bằng — nhưng hoà theo từ điển chỉ xét trong các chuỗi độ dài `minSize` là cách hiểu hợp lý (ta quy ước như vậy). (2) Dùng `Counter`. (3) Sắp theo khoá `(-count, chuỗi)`.

**Bài 5:** (1) Tiền tố có thể có ít ký tự phân biệt hơn ⇒ phải thử mọi độ dài. (2) Nhưng **hậu tố/tiền tố của chuỗi dài** không giúp gì; brute force theo độ dài vẫn ổn vì `maxSize <= 26`. (3) Với mỗi `L`, chạy cửa sổ trượt độ dài `L` và đếm.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
assert Solution().maxFreq("abcabc", 3, 3, 3) == 2
assert Solution().maxFreq("abcabc", 2, 3, 3) == 0
# Bài 2
s = "aabaab"
wins = [(i, s[i:i + 2]) for i in range(len(s) - 1) if len(set(s[i:i + 2])) <= 1]
print("Bài 2 cửa sổ hợp lệ:", wins)
assert Solution().maxFreq(s, 1, 2, 3) == 2

# Bài 3
def count_distinct_windows(s, L):
    freq = [0] * 26
    distinct = 0
    res = []
    for r, ch in enumerate(s):
        k = ord(ch) - 97
        if freq[k] == 0:
            distinct += 1
        freq[k] += 1
        if r >= L:
            ko = ord(s[r - L]) - 97
            freq[ko] -= 1
            if freq[ko] == 0:
                distinct -= 1
        if r >= L - 1:
            res.append(distinct)
    return res

assert count_distinct_windows("aababcaab", 3) == [2, 2, 2, 3, 3, 2, 2]
rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 15)))
    L = rng.randint(1, len(s))
    assert count_distinct_windows(s, L) == [len(set(s[i:i + L])) for i in range(len(s) - L + 1)]
print("Bài 1–3: OK")

Bài 2 cửa sổ hợp lệ: [(0, 'aa'), (3, 'aa')]
Bài 1–3: OK


In [34]:
# Bài 4
def best_substring(s, maxLetters, minSize, maxSize):
    cnt = Counter(s[i:i + minSize] for i in range(len(s) - minSize + 1)
                  if len(set(s[i:i + minSize])) <= maxLetters)
    if not cnt:
        return ""
    return min(cnt, key=lambda t: (-cnt[t], t))

assert best_substring("aababcaab", 2, 3, 4) == "aab"
assert best_substring("aaaa", 1, 3, 3) == "aaa"
assert best_substring("abcde", 2, 3, 3) == ""
assert best_substring("abab", 2, 2, 2) == "ab"     # "ab" x2, "ba" x1

# Bài 5
def max_freq_at_least(s, k, minSize, maxSize):
    best = 0
    for L in range(minSize, maxSize + 1):
        freq = [0] * 26
        distinct = 0
        cnt = {}
        for r, ch in enumerate(s):
            c = ord(ch) - 97
            if freq[c] == 0:
                distinct += 1
            freq[c] += 1
            if r >= L:
                co = ord(s[r - L]) - 97
                freq[co] -= 1
                if freq[co] == 0:
                    distinct -= 1
            if r >= L - 1 and distinct >= k:
                t = s[r - L + 1:r + 1]
                cnt[t] = cnt.get(t, 0) + 1
                best = max(best, cnt[t])
    return best

rng = random.Random(11)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 14)))
    mn = rng.randint(1, len(s)); mx = rng.randint(mn, len(s)); k = rng.randint(1, 4)
    assert max_freq_at_least(s, k, mn, mx) == at_least_brute(s, k, mn, mx)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ độ dài cố định `L` + số ký tự phân biệt:**

```python
freq, distinct = [0] * 26, 0
for r, ch in enumerate(s):
    k = ord(ch) - 97
    if freq[k] == 0: distinct += 1
    freq[k] += 1
    if r >= L:                                # bỏ ký tự rơi khỏi cửa sổ
        ko = ord(s[r - L]) - 97
        freq[ko] -= 1
        if freq[ko] == 0: distinct -= 1
    if r >= L - 1:                            # cửa sổ s[r-L+1 : r+1] đã đủ
        ...  # xử lý cửa sổ
```

**Nhớ:** bài 1297 ⇒ `L = minSize`, bỏ qua `maxSize`.

**Checklist trước khi submit:**
- [ ] Chỉ xét độ dài `minSize` (đã hiểu vì sao đúng).
- [ ] Số cửa sổ là `n − L + 1` (không thiếu cửa sổ cuối).
- [ ] Đếm **chồng lấn** — không dùng `str.count`.
- [ ] `distinct` giảm khi `freq` về 0.
- [ ] Trả `0` khi không có cửa sổ hợp lệ (`default=0` hoặc `best = 0`).
- [ ] Đúng chữ ký: `maxFreq(self, s: str, maxLetters: int, minSize: int, maxSize: int) -> int`.